# Evaluating Fine-Tuned scGPT on Norman 2019: GI-Stratified Metrics

**Goal**: Load saved scGPT predictions on Norman 2019 test set, evaluate using standard metrics (MSE, PCC) *and* stratified by genetic interaction (GI) type.

## Evaluation Framework

### Standard metrics (per combinatorial perturbation):
- **MSE_top20**: Mean squared error on top 20 differentially expressed genes
- **PCC_delta**: Pearson correlation on Δexpression (predicted vs. observed change from control)

### GI-stratified evaluation (our contribution):
- Classify each combo perturbation by genetic interaction type
- Report metrics separately for additive / synergistic / epistatic / neomorphic

### Norman et al. (2019) interaction decomposition:
For a two-gene perturbation (A+B), the "manifold" linear regression:

$$\delta_{ab} = c_1 \delta_a + c_2 \delta_b + \epsilon$$

where:
- $\delta_a, \delta_b$ = single-gene perturbation effects (mean shift from control)
- $c_1, c_2$ = regression coefficients (dominance weights)
- $\epsilon$ = residual = **neomorphic** effects (new biology from combination)
- Large $|\epsilon|$ → non-additive interaction

**GI Score** (scalar): $\text{GI} = \delta_{ab} - (\delta_a + \delta_b)$

### References
- Norman et al., *Science* (2019). DOI: 10.1126/science.aax4438
- Roohani et al. (GEARS), *Nature Biotechnology* (2024). DOI: 10.1038/s41587-023-02055-5
- Cui et al. (scGPT), *Nature Methods* (2024). DOI: 10.1038/s41592-024-02201-0
- Liu et al. (scISP), *bioRxiv* (2025). DOI: 10.1101/2025.05.11.653338

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from scipy import sparse
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

## 1. Load Data

We need:
1. **Ground truth adata** with split labels (your pre-split file)
2. **scGPT predictions** (saved from training — typically a dict: `{condition: predicted_expression}`)

Adjust paths below to match your directory structure.

In [2]:
# === CONFIGURE PATHS ===
# Path to your pre-split Norman h5ad
ADATA_PATH = "../../data/splits/norman_2019_raw_sparse_with_splits_original_from_GEARS.h5ad"

# Path to scGPT saved predictions
# After fine-tuning, scGPT tutorial saves results as:
#   - A dict of {perturbation_condition: predicted_mean_expression}
#   - Or a pickle/npz file
# Adjust based on how your training script saved outputs.
PRED_PATH = "../../results/scgpt_norman/best_model_predictions.npy"  # example

# Alternative: if scGPT saved an adata with .obsm['predicted']
# PRED_ADATA_PATH = "../../results/scgpt_norman/pred_adata.h5ad"

In [3]:
# Load ground truth
adata = sc.read_h5ad(ADATA_PATH)
print(f"Loaded adata: {adata.shape}")
print(f"Split distribution:\n{adata.obs['gears_split'].value_counts()}")
print(f"\nSample conditions: {adata.obs['condition'].unique()[:10]}")

Loaded adata: (89357, 5045)
Split distribution:
gears_split
train    47192
test     30969
val      11196
Name: count, dtype: int64

Sample conditions: ['TSC22D1+ctrl', 'KLF1+MAP2K6', 'ctrl', 'CEBPE+RUNX1T1', 'MAML2+ctrl', 'ctrl+CEBPE', 'CBL+PTPN9', 'TGFBR2+ETS2', 'SGK1+TBX3', 'DUSP9+ctrl']
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']


### 1.1 Prepare expression data

scGPT evaluates on log1p-normalized data. If your adata is raw counts, normalize here.

In [4]:
# Check if already normalized (raw counts are integers, normalized are floats)
X_sample = adata.X[:5].toarray() if sparse.issparse(adata.X) else adata.X[:5]
is_raw = np.allclose(X_sample, X_sample.astype(int))
print(f"Data appears to be {'raw counts' if is_raw else 'already normalized'}")

if is_raw:
    # Store raw in .raw, then normalize
    adata.raw = adata.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("Applied normalize_total + log1p")

Data appears to be raw counts
Applied normalize_total + log1p


## 2. Compute Ground Truth Expression Profiles

For each perturbation condition, compute:
- **Mean expression** across cells with that condition
- **Δexpression** = mean(perturbed) - mean(control)

In [5]:
# Identify control cells
# In Norman/GEARS, control condition is typically 'ctrl' or 'non-targeting' or 'ctrl+ctrl'
control_labels = ['ctrl', 'control', 'non-targeting', 'ctrl+ctrl']
ctrl_mask = adata.obs['condition'].isin(control_labels)
print(f"Control cells found: {ctrl_mask.sum()}")

if ctrl_mask.sum() == 0:
    # Try pattern matching
    ctrl_mask = adata.obs['condition'].str.lower().str.contains('ctrl')
    print(f"Control cells (pattern match): {ctrl_mask.sum()}")

# Compute control mean expression
X_ctrl = adata[ctrl_mask].X
if sparse.issparse(X_ctrl):
    X_ctrl = X_ctrl.toarray()
ctrl_mean = X_ctrl.mean(axis=0).flatten()

Control cells found: 7353


In [6]:
def get_condition_mean(adata, condition):
    """Get mean expression for a given perturbation condition."""
    mask = adata.obs['condition'] == condition
    if mask.sum() == 0:
        return None
    X = adata[mask].X
    if sparse.issparse(X):
        X = X.toarray()
    return X.mean(axis=0).flatten()


# Identify test set combinatorial perturbations
test_mask = adata.obs['gears_split'] == 'test'
test_conditions = adata.obs.loc[test_mask, 'condition'].unique()

# Filter to combo perturbations (contain '+' and both genes are non-ctrl)
combo_conditions = []
for cond in test_conditions:
    if '+' in cond:
        genes = cond.split('+')
        if len(genes) == 2 and 'ctrl' not in genes[0].lower() and 'ctrl' not in genes[1].lower():
            combo_conditions.append(cond)

print(f"Test set combo perturbations: {len(combo_conditions)}")
print(f"Examples: {combo_conditions[:5]}")

Test set combo perturbations: 77
Examples: ['KLF1+MAP2K6', 'TGFBR2+ETS2', 'SGK1+TBX3', 'MAP2K6+SPI1', 'UBASH3B+PTPN12']


In [7]:
# Compute ground truth profiles for all relevant conditions
gt_profiles = {}  # condition -> mean expression vector
gt_delta = {}     # condition -> delta from control

# All conditions (including singles needed for GI computation)
all_conditions = set(adata.obs['condition'].unique())

for cond in all_conditions:
    mean_expr = get_condition_mean(adata, cond)
    if mean_expr is not None:
        gt_profiles[cond] = mean_expr
        gt_delta[cond] = mean_expr - ctrl_mean

print(f"Computed profiles for {len(gt_profiles)} conditions")

Computed profiles for 277 conditions


## 3. Load scGPT Predictions

scGPT's perturbation tutorial typically saves predictions as:
- A dict mapping condition → predicted mean expression
- Or a results dict with keys like `'test_res'`

**Adapt this section to match your saved format.**

In [8]:
# === OPTION A: Load from numpy/pickle ===
# Uncomment and adapt based on your save format

# import pickle
# with open(PRED_PATH, 'rb') as f:
#     pred_results = pickle.load(f)

# === OPTION B: Load from scGPT's default output ===
# scGPT perturbation tutorial typically stores:
#   results = {
#       condition: {
#           'pred': np.array (predicted mean expression),
#           'truth': np.array (ground truth mean expression),
#           'de_genes': list (top 20 DE gene indices),
#       }
#   }

# === OPTION C: Re-run inference from saved checkpoint ===
# If predictions weren't saved, you need to load the model checkpoint
# and run inference. See Section 3.1 below.

# For now, placeholder — replace with actual loading:
pred_results = {}  # TODO: load your saved predictions
print(f"Loaded predictions for {len(pred_results)} conditions")

Loaded predictions for 0 conditions


### 3.1 (Optional) Re-run inference from checkpoint

If you only saved the model checkpoint (not predictions), run inference here.
This requires the scGPT environment and model code.

In [9]:
# === Uncomment if you need to re-run inference ===

# import sys
# sys.path.insert(0, '/path/to/scGPT')  # adjust
# from scgpt.model import TransformerModel
# from scgpt.tokenizer import GeneVocab
# import torch
#
# CHECKPOINT_PATH = "../../results/scgpt_norman/best_model.pt"
# VOCAB_PATH = "../../models/scGPT_human/vocab.json"
#
# vocab = GeneVocab.from_file(VOCAB_PATH)
# model = ...  # load model architecture + checkpoint
# model.eval()
#
# # Run inference on test conditions
# # (refer to scGPT Tutorial_Perturbation.ipynb evaluation cells)

## 4. Standard Evaluation Metrics

### 4.1 Per-perturbation MSE (top-20 DE genes) and PCC (delta expression)

Following GEARS/scGPT convention:
- **Top 20 DE genes**: For each perturbation, select the 20 genes with largest absolute Δexpression in ground truth
- **MSE**: computed only on those 20 genes
- **PCC**: Pearson correlation on delta expression across all genes (or top-20)

In [10]:
def get_top_de_genes(delta_expr, n=20):
    """Return indices of top n DE genes by absolute magnitude."""
    return np.argsort(np.abs(delta_expr))[::-1][:n]


def compute_mse_top20(pred_delta, true_delta, n=20):
    """MSE on top-20 DE genes (by ground truth magnitude)."""
    top_idx = get_top_de_genes(true_delta, n)
    return np.mean((pred_delta[top_idx] - true_delta[top_idx]) ** 2)


def compute_pcc_delta(pred_delta, true_delta):
    """Pearson correlation on delta expression."""
    # Avoid constant vectors
    if np.std(pred_delta) < 1e-10 or np.std(true_delta) < 1e-10:
        return 0.0
    r, _ = pearsonr(pred_delta, true_delta)
    return r


def compute_pcc_top20(pred_delta, true_delta, n=20):
    """PCC on top-20 DE genes only."""
    top_idx = get_top_de_genes(true_delta, n)
    return compute_pcc_delta(pred_delta[top_idx], true_delta[top_idx])

In [11]:
# Compute metrics for each test combo perturbation
results_list = []

for cond in combo_conditions:
    # Ground truth delta
    if cond not in gt_delta:
        continue
    true_delta = gt_delta[cond]
    
    # Predicted delta
    if cond not in pred_results:
        continue
    # Adapt extraction based on your pred_results format:
    pred_expr = pred_results[cond]  # or pred_results[cond]['pred']
    pred_delta = pred_expr - ctrl_mean  # if pred is absolute expression
    # pred_delta = pred_results[cond]  # if pred is already delta
    
    gene_a, gene_b = cond.split('+')
    
    results_list.append({
        'condition': cond,
        'gene_a': gene_a,
        'gene_b': gene_b,
        'mse_top20': compute_mse_top20(pred_delta, true_delta),
        'pcc_delta': compute_pcc_delta(pred_delta, true_delta),
        'pcc_top20': compute_pcc_top20(pred_delta, true_delta),
        'n_cells': (adata.obs['condition'] == cond).sum(),
    })

results_df = pd.DataFrame(results_list)
print(f"Evaluated {len(results_df)} combo perturbations")
print(f"\nAggregate metrics:")
print(f"  MSE_top20:  {results_df['mse_top20'].mean():.4f} ± {results_df['mse_top20'].std():.4f}")
print(f"  PCC_delta:  {results_df['pcc_delta'].mean():.4f} ± {results_df['pcc_delta'].std():.4f}")
print(f"  PCC_top20:  {results_df['pcc_top20'].mean():.4f} ± {results_df['pcc_top20'].std():.4f}")

Evaluated 0 combo perturbations

Aggregate metrics:


KeyError: 'mse_top20'

## 5. Genetic Interaction (GI) Classification

### 5.1 Compute GI scores

For each combo perturbation A+B:

$$\text{GI}_{\text{gene } g} = \delta_{ab,g} - (\delta_{a,g} + \delta_{b,g})$$

where $\delta_{x,g}$ is the expression change of gene $g$ under perturbation $x$.

Aggregate GI per perturbation pair: use the **magnitude of the GI vector** across top DE genes.

### 5.2 Norman's manifold decomposition

Additionally, fit the linear model per combo:

$$\delta_{ab} = c_1 \delta_a + c_2 \delta_b + \epsilon$$

- $|\epsilon|$ large → neomorphic effects
- $c_1 \gg c_2$ → gene A dominant
- $c_1 \approx c_2 \approx 1$ → purely additive

In [ ]:
def compute_gi_score(delta_ab, delta_a, delta_b, top_n=20):
    """
    Compute genetic interaction score.
    
    GI = observed(A+B) - additive_expectation(A+B)
    where additive_expectation = delta_a + delta_b
    
    Returns:
        gi_vector: per-gene GI scores
        gi_magnitude: L2 norm of GI on top DE genes
        gi_mean: mean GI across top DE genes (signed)
    """
    additive_expect = delta_a + delta_b
    gi_vector = delta_ab - additive_expect
    
    # Focus on top DE genes of the combo perturbation
    top_idx = get_top_de_genes(delta_ab, top_n)
    gi_top = gi_vector[top_idx]
    
    return {
        'gi_vector': gi_vector,
        'gi_magnitude': np.linalg.norm(gi_top),
        'gi_mean_signed': np.mean(gi_top),
        'gi_frac_variance': np.var(gi_top) / (np.var(delta_ab[top_idx]) + 1e-10),
    }


def norman_manifold_decomposition(delta_ab, delta_a, delta_b):
    """
    Norman et al. manifold decomposition:
    delta_ab = c1 * delta_a + c2 * delta_b + epsilon
    
    Uses linear regression (OLS) to fit c1, c2.
    epsilon = residual = neomorphic component.
    
    Returns:
        c1, c2: regression coefficients
        epsilon: residual vector
        r2: R-squared (fraction of variance explained by additive model)
        epsilon_magnitude: L2 norm of epsilon
    """
    X = np.column_stack([delta_a, delta_b])
    y = delta_ab
    
    reg = LinearRegression(fit_intercept=False)
    reg.fit(X, y)
    
    c1, c2 = reg.coef_
    y_pred = reg.predict(X)
    epsilon = y - y_pred
    
    ss_res = np.sum(epsilon ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2) + 1e-10
    r2 = 1 - ss_res / ss_tot
    
    return {
        'c1': c1,
        'c2': c2,
        'epsilon': epsilon,
        'r2': r2,
        'epsilon_magnitude': np.linalg.norm(epsilon),
        'epsilon_top20_magnitude': np.linalg.norm(
            epsilon[get_top_de_genes(delta_ab, 20)]
        ),
    }

In [ ]:
# Compute GI scores and manifold decomposition for each combo
gi_data = []

for cond in combo_conditions:
    gene_a, gene_b = cond.split('+')
    
    # Need single-gene deltas
    # In Norman/GEARS format, single perturbations are "GENE+ctrl" or just "GENE"
    single_a_keys = [f"{gene_a}+ctrl", f"ctrl+{gene_a}", gene_a]
    single_b_keys = [f"{gene_b}+ctrl", f"ctrl+{gene_b}", gene_b]
    
    delta_a = None
    for k in single_a_keys:
        if k in gt_delta:
            delta_a = gt_delta[k]
            break
    
    delta_b = None
    for k in single_b_keys:
        if k in gt_delta:
            delta_b = gt_delta[k]
            break
    
    if delta_a is None or delta_b is None or cond not in gt_delta:
        continue
    
    delta_ab = gt_delta[cond]
    
    # GI score
    gi = compute_gi_score(delta_ab, delta_a, delta_b)
    
    # Manifold decomposition
    manifold = norman_manifold_decomposition(delta_ab, delta_a, delta_b)
    
    gi_data.append({
        'condition': cond,
        'gene_a': gene_a,
        'gene_b': gene_b,
        'gi_magnitude': gi['gi_magnitude'],
        'gi_mean_signed': gi['gi_mean_signed'],
        'gi_frac_variance': gi['gi_frac_variance'],
        'c1': manifold['c1'],
        'c2': manifold['c2'],
        'r2_additive': manifold['r2'],
        'epsilon_magnitude': manifold['epsilon_magnitude'],
        'epsilon_top20_mag': manifold['epsilon_top20_magnitude'],
    })

gi_df = pd.DataFrame(gi_data)
print(f"Computed GI for {len(gi_df)} combo perturbations")
gi_df.head()

### 5.3 Classify interaction types

Classification based on GI magnitude and manifold R²:

| Type | Criterion | Interpretation |
|------|-----------|----------------|
| **Additive** | small GI magnitude, high R² | Effect is sum of parts |
| **Synergistic** | GI > 0 (positive, large) | Super-additive |
| **Epistatic/Buffering** | GI < 0 (negative, large) | Sub-additive |
| **Neomorphic** | Large ε, low R² | Qualitatively new effects |

We use Norman's own annotations where available, and compute classifications for the remainder.

In [ ]:
def classify_gi_type(row, gi_threshold_quantile=0.75):
    """
    Classify interaction type based on GI metrics.
    
    Thresholds are relative (quantile-based) since absolute scale
    depends on normalization.
    """
    # Will be set after computing all GI scores
    return row  # placeholder


# Compute thresholds from distribution
gi_mag_threshold = gi_df['gi_magnitude'].quantile(0.75)
r2_threshold = 0.5  # below this = poorly explained by additive model

def assign_gi_type(row):
    """Assign GI type based on magnitude, sign, and R²."""
    if row['r2_additive'] < r2_threshold and row['epsilon_top20_mag'] > gi_mag_threshold:
        return 'neomorphic'
    elif row['gi_magnitude'] < gi_df['gi_magnitude'].quantile(0.25):
        return 'additive'
    elif row['gi_mean_signed'] > 0:
        return 'synergistic'
    else:
        return 'suppressive'

gi_df['gi_type'] = gi_df.apply(assign_gi_type, axis=1)
print("GI type distribution:")
print(gi_df['gi_type'].value_counts())

In [ ]:
# === OPTIONAL: Use Norman's published annotations ===
# If you have the Norman supplementary table with interaction type labels,
# load and merge here instead of computing from scratch.

# norman_annotations = pd.read_csv('norman_2019_interaction_annotations.csv')
# gi_df = gi_df.merge(norman_annotations[['condition', 'interaction_type_norman']], 
#                      on='condition', how='left')
# gi_df['gi_type'] = gi_df['interaction_type_norman'].fillna(gi_df['gi_type'])

## 6. GI-Stratified Evaluation

**This is the core novel analysis.** Report prediction quality stratified by interaction type.

In [ ]:
# Merge GI classifications with prediction results
eval_df = results_df.merge(gi_df[['condition', 'gi_type', 'gi_magnitude', 
                                   'r2_additive', 'epsilon_top20_mag']], 
                           on='condition', how='inner')

print(f"Perturbations with both predictions and GI labels: {len(eval_df)}")

In [ ]:
# Stratified metrics table
stratified = eval_df.groupby('gi_type').agg(
    n=('condition', 'count'),
    mse_top20_mean=('mse_top20', 'mean'),
    mse_top20_std=('mse_top20', 'std'),
    pcc_delta_mean=('pcc_delta', 'mean'),
    pcc_delta_std=('pcc_delta', 'std'),
    pcc_top20_mean=('pcc_top20', 'mean'),
    pcc_top20_std=('pcc_top20', 'std'),
).round(4)

# Add overall row
overall = pd.DataFrame({
    'n': [len(eval_df)],
    'mse_top20_mean': [eval_df['mse_top20'].mean()],
    'mse_top20_std': [eval_df['mse_top20'].std()],
    'pcc_delta_mean': [eval_df['pcc_delta'].mean()],
    'pcc_delta_std': [eval_df['pcc_delta'].std()],
    'pcc_top20_mean': [eval_df['pcc_top20'].mean()],
    'pcc_top20_std': [eval_df['pcc_top20'].std()],
}, index=['OVERALL']).round(4)

stratified_table = pd.concat([stratified, overall])
print("\n=== scGPT (fine-tuned) — Stratified by GI Type ===")
print(stratified_table.to_string())

## 7. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 7a. MSE by GI type
order = ['additive', 'synergistic', 'suppressive', 'neomorphic']
order = [o for o in order if o in eval_df['gi_type'].values]  # only present types

sns.boxplot(data=eval_df, x='gi_type', y='mse_top20', order=order, ax=axes[0],
            palette='Set2')
axes[0].set_title('MSE (top-20 DE) by GI Type')
axes[0].set_xlabel('Genetic Interaction Type')
axes[0].set_ylabel('MSE')

# 7b. PCC by GI type
sns.boxplot(data=eval_df, x='gi_type', y='pcc_top20', order=order, ax=axes[1],
            palette='Set2')
axes[1].set_title('PCC (top-20 DE) by GI Type')
axes[1].set_xlabel('Genetic Interaction Type')
axes[1].set_ylabel('Pearson r')

# 7c. GI magnitude vs prediction error
axes[2].scatter(eval_df['gi_magnitude'], eval_df['mse_top20'], 
               c=eval_df['gi_type'].map({'additive': 'C0', 'synergistic': 'C1', 
                                          'suppressive': 'C2', 'neomorphic': 'C3'}),
               alpha=0.7, edgecolors='k', linewidths=0.5)
axes[2].set_xlabel('GI Magnitude (||ε||)')
axes[2].set_ylabel('MSE (top-20 DE)')
axes[2].set_title('Non-Additivity vs Prediction Error')

plt.tight_layout()
plt.savefig('scgpt_gi_stratified_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 7d. Manifold decomposition scatter: c1 vs c2 colored by prediction quality
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

merged = eval_df.merge(gi_df[['condition', 'c1', 'c2']], on='condition')

sc1 = axes[0].scatter(merged['c1'], merged['c2'], c=merged['mse_top20'],
                      cmap='RdYlGn_r', alpha=0.8, edgecolors='k', linewidths=0.5)
axes[0].plot([0, 1.5], [0, 1.5], 'k--', alpha=0.3, label='c1=c2 (symmetric)')
axes[0].axhline(1, color='gray', ls=':', alpha=0.3)
axes[0].axvline(1, color='gray', ls=':', alpha=0.3)
axes[0].set_xlabel('c1 (gene A dominance)')
axes[0].set_ylabel('c2 (gene B dominance)')
axes[0].set_title('Manifold Coefficients → Prediction Error')
plt.colorbar(sc1, ax=axes[0], label='MSE top-20')

sc2 = axes[1].scatter(merged['r2_additive'], merged['mse_top20'],
                      c=merged['gi_type'].map({'additive': 'C0', 'synergistic': 'C1', 
                                               'suppressive': 'C2', 'neomorphic': 'C3'}),
                      alpha=0.7, edgecolors='k', linewidths=0.5)
axes[1].set_xlabel('R² (additive model fit)')
axes[1].set_ylabel('MSE (top-20 DE)')
axes[1].set_title('Additive Fit Quality vs Prediction Error')

plt.tight_layout()
plt.savefig('scgpt_manifold_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. GEARS-Split Stratification (seen 0/1/2)

In addition to GI-type stratification, also report by **seen level**:
- **seen2**: both single genes seen in training
- **seen1**: one gene seen, one unseen
- **seen0**: both genes unseen

This is the standard GEARS split stratification.

In [ ]:
# Determine seen level from split
train_conditions = set(adata.obs.loc[adata.obs['gears_split'] == 'train', 'condition'].unique())

# Extract single-gene perturbations seen in training
train_singles = set()
for cond in train_conditions:
    if '+' in cond:
        for g in cond.split('+'):
            if g.lower() != 'ctrl':
                train_singles.add(g)
    else:
        train_singles.add(cond)

def get_seen_level(gene_a, gene_b):
    seen = int(gene_a in train_singles) + int(gene_b in train_singles)
    return f'seen{seen}'

eval_df['seen_level'] = eval_df.apply(
    lambda r: get_seen_level(r['gene_a'], r['gene_b']), axis=1)

# Stratified by seen level
seen_stratified = eval_df.groupby('seen_level').agg(
    n=('condition', 'count'),
    mse_top20_mean=('mse_top20', 'mean'),
    pcc_delta_mean=('pcc_delta', 'mean'),
    pcc_top20_mean=('pcc_top20', 'mean'),
).round(4)

print("\n=== scGPT — Stratified by Seen Level ===")
print(seen_stratified.to_string())

## 9. Cross-Tabulation: Seen Level × GI Type

The most informative view: does non-additivity compound with novelty?

In [ ]:
# Heatmap: MSE by (seen_level × gi_type)
cross_mse = eval_df.pivot_table(
    values='mse_top20', index='seen_level', columns='gi_type', aggfunc='mean'
)
cross_pcc = eval_df.pivot_table(
    values='pcc_top20', index='seen_level', columns='gi_type', aggfunc='mean'
)
cross_n = eval_df.pivot_table(
    values='condition', index='seen_level', columns='gi_type', aggfunc='count'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cross_mse, annot=True, fmt='.3f', cmap='Reds', ax=axes[0])
axes[0].set_title('Mean MSE (top-20) by Seen Level × GI Type')

sns.heatmap(cross_pcc, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[1],
            vmin=0, vmax=1)
axes[1].set_title('Mean PCC (top-20) by Seen Level × GI Type')

plt.tight_layout()
plt.savefig('scgpt_cross_tab_eval.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSample sizes per cell:")
print(cross_n.to_string())

## 10. Additive Baseline Comparison

Compare scGPT against a simple **additive baseline**:

$$\hat{\delta}_{ab} = \delta_a + \delta_b$$

If scGPT doesn't beat this on non-additive cases, the fine-tuning isn't learning interactions.

In [ ]:
# Compute additive baseline predictions
additive_results = []

for cond in combo_conditions:
    gene_a, gene_b = cond.split('+')
    
    # Find single-gene deltas
    delta_a, delta_b, delta_ab = None, None, None
    for k in [f"{gene_a}+ctrl", f"ctrl+{gene_a}", gene_a]:
        if k in gt_delta:
            delta_a = gt_delta[k]
            break
    for k in [f"{gene_b}+ctrl", f"ctrl+{gene_b}", gene_b]:
        if k in gt_delta:
            delta_b = gt_delta[k]
            break
    if cond in gt_delta:
        delta_ab = gt_delta[cond]
    
    if delta_a is None or delta_b is None or delta_ab is None:
        continue
    
    # Additive prediction
    pred_additive = delta_a + delta_b
    
    additive_results.append({
        'condition': cond,
        'mse_top20_additive': compute_mse_top20(pred_additive, delta_ab),
        'pcc_delta_additive': compute_pcc_delta(pred_additive, delta_ab),
        'pcc_top20_additive': compute_pcc_top20(pred_additive, delta_ab),
    })

additive_df = pd.DataFrame(additive_results)

# Merge with scGPT results
comparison = eval_df.merge(additive_df, on='condition', how='inner')

print("\n=== Additive Baseline vs scGPT ===")
for gi_type in comparison['gi_type'].unique():
    sub = comparison[comparison['gi_type'] == gi_type]
    print(f"\n{gi_type} (n={len(sub)}):")
    print(f"  MSE — Additive: {sub['mse_top20_additive'].mean():.4f}  |  scGPT: {sub['mse_top20'].mean():.4f}")
    print(f"  PCC — Additive: {sub['pcc_top20_additive'].mean():.4f}  |  scGPT: {sub['pcc_top20'].mean():.4f}")

In [ ]:
# Scatter: additive MSE vs scGPT MSE (colored by GI type)
fig, ax = plt.subplots(figsize=(6, 6))

color_map = {'additive': 'C0', 'synergistic': 'C1', 'suppressive': 'C2', 'neomorphic': 'C3'}
for gi_type, sub in comparison.groupby('gi_type'):
    ax.scatter(sub['mse_top20_additive'], sub['mse_top20'],
              label=gi_type, alpha=0.7, c=color_map.get(gi_type, 'gray'),
              edgecolors='k', linewidths=0.5)

# Diagonal: below = scGPT better, above = additive better
lim = max(comparison['mse_top20'].max(), comparison['mse_top20_additive'].max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3)
ax.set_xlabel('Additive Baseline MSE')
ax.set_ylabel('scGPT MSE')
ax.set_title('scGPT vs Additive Baseline\n(below diagonal = scGPT wins)')
ax.legend()
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)

plt.tight_layout()
plt.savefig('scgpt_vs_additive.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Export Results

Save evaluation tables for comparison with other models (GEARS, CPA, etc.).

In [ ]:
# Save per-perturbation results
comparison.to_csv('scgpt_eval_per_perturbation.csv', index=False)

# Save stratified summary
stratified_table.to_csv('scgpt_eval_stratified_by_gi.csv')

# Save GI annotations (reusable across models)
gi_df.to_csv('norman_gi_annotations.csv', index=False)

print("Saved:")
print("  scgpt_eval_per_perturbation.csv")
print("  scgpt_eval_stratified_by_gi.csv")
print("  norman_gi_annotations.csv")

---

## Summary

### Expected findings pattern

| GI Type | Additive Baseline | scGPT (fine-tuned) | Interpretation |
|---------|-------------------|--------------------|----------------|
| Additive | Good | Good | Both handle simple cases |
| Synergistic | Poor | Moderate? | scGPT may partially learn from data |
| Suppressive | Poor | Moderate? | Similar |
| Neomorphic | Very poor | Poor? | Hardest case — motivates architectural changes |

If scGPT matches or barely beats the additive baseline on non-additive cases,
this directly motivates the multiplicative attention modification (Stage 2).

### Next steps
1. Run this notebook with actual scGPT predictions
2. Repeat for GEARS baseline (same evaluation code, different predictions)
3. Repeat for CPA baseline
4. Compile stratified comparison table across all models
5. Proceed to Track A (scGPT + M matrix) if pattern confirms hypothesis